In [3]:
import torch
from torchvision import transforms
from torch.utils.data import DataLoader
from torchvision import models
import torch.nn as nn

In [4]:
!pip install gdown
import gdown

# Replace 'YOUR_FILE_ID' with your actual ID
file_id = '1UrTmhu2nLvWMv-AMfoa3Z6JBZwalagA7'
url = f'https://drive.google.com/uc?id={file_id}'
output = 'data.zip' # or 'dataset.csv'

gdown.download(url, output, quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=1UrTmhu2nLvWMv-AMfoa3Z6JBZwalagA7
From (redirected): https://drive.google.com/uc?id=1UrTmhu2nLvWMv-AMfoa3Z6JBZwalagA7&confirm=t&uuid=f4edbaa6-6ab0-4216-951b-241f438a15f5
To: /kaggle/working/data.zip
100%|██████████| 251M/251M [00:01<00:00, 215MB/s]  


'data.zip'

In [5]:
# -q makes it "quiet" (hides the list of every file being unzipped)
# -d specifies the destination directory
!unzip -q /kaggle/working/data.zip -d /kaggle/working/my_data


In [6]:
train_path = "/kaggle/working/my_data/content/drive/MyDrive/datasets/data/train"
val_path   = "/kaggle/working/my_data/content/drive/MyDrive/datasets/data/val"

In [7]:
train_transforms=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406], # ImageNet means
        std=[0.229, 0.224, 0.225]   # ImageNet stds
    )
])
val_test_transforms=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406], # ImageNet means
        std=[0.229, 0.224, 0.225]   # ImageNet stds
    )
])

In [8]:
from torchvision import datasets
train_data=datasets.ImageFolder(train_path,transform=train_transforms)
val_data=datasets.ImageFolder(val_path,transform=test_transforms)

In [25]:
import torch
from torch.utils.data import random_split, DataLoader


train_loader = DataLoader(train_data, batch_size=64, shuffle=True)

total_val_images = len(val_data)
test_size = total_val_images // 2  
new_val_size = total_val_images - test_size 


generator = torch.Generator().manual_seed(42) 
new_val_data, test_data = random_split(val_data, [new_val_size, test_size], generator=generator)

# 4. Create the separate DataLoaders
val_loader = DataLoader(new_val_data, batch_size=64, shuffle=False)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

print(f"Total Images in physical 'val' folder: {total_val_images}")
print(f"--> Assigned to Validation Loader: {len(new_val_data)}")
print(f"--> Assigned to Test Loader: {len(test_data)}")

Total Images in physical 'val' folder: 3106
--> Assigned to Validation Loader: 1553
--> Assigned to Test Loader: 1553


In [ ]:
# import torch.nn as nn
# class trash_classifier(nn.Module):
#   def __init__(self):
#     super().__init__()
#     self.conv=nn.Sequential(
#         nn.Conv2d(in_channels=3,out_channels=16,kernel_size=3),#Shape(128 -> 126) actual 224 X 224 X 3
#         nn.ReLU(),#Shape (222)
#         nn.MaxPool2d(2,2),#Shape (222 -> 111) so 222 X 222 X 16

#         nn.Conv2d(in_channels=16,out_channels=32,kernel_size=3,padding=1),#Shape (111 -> 111 ) as we use padding there , so 222 X 222 X 16.
#         nn.ReLU(),
#         nn.MaxPool2d(2,2), # shape (111 -> 55) so 55 X 55 X 32

#         nn.Conv2d(32, 64, 3, padding=1),
#         nn.BatchNorm2d(64),
#         nn.ReLU(),
#         nn.MaxPool2d(2,2),

#         nn.Conv2d(64,128, 3, padding=1),
#         nn.ReLU(),
#         nn.MaxPool2d(2,2),


#     )
#     self.fc=nn.Sequential(
#         nn.Flatten(),
#         nn.Linear(7*7*128,200),
#         nn.ReLU(),
#         nn.Linear(200,12)
#     )
#   def forward(self,x):
#       x=self.conv(x)
#       x=self.fc(x)
#       return x


In [10]:
model=models.resnet18(pretrained=True)
model.fc=nn.Linear(model.fc.in_features,12)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 146MB/s]


In [11]:
for param in model.parameters():
    param.requires_grad=False
    
for param in model.fc.parameters():
    param.requires_grad=True

In [12]:

loss_fn= nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=0.001,weight_decay=1e-4)


In [26]:
for epoch in range(5):
  model.train()
  total_loss=0
  for (images,labels) in train_loader:
    outputs=model(images)
    loss=loss_fn(outputs,labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    total_loss+=loss.item()
  current_lr=optimizer.param_groups[0]['lr']
  avg_loss = total_loss / len(train_loader)
  print(f"Epoch {epoch+1}/5, Loss: {avg_loss:.4f}, Learning Rate = {current_lr:.6f}")

Epoch 1/5, Loss: 0.2699, Learning Rate = 0.001000
Epoch 2/5, Loss: 0.2608, Learning Rate = 0.001000
Epoch 3/5, Loss: 0.2505, Learning Rate = 0.001000
Epoch 4/5, Loss: 0.2488, Learning Rate = 0.001000
Epoch 5/5, Loss: 0.2361, Learning Rate = 0.001000


In [27]:
train_correct = 0
train_total = 0

for images, labels in train_loader:
    outputs = model(images)
    _, preds = torch.max(outputs, 1)

    train_correct += (preds == labels).sum().item()
    train_total += labels.size(0)

train_acc = train_correct / train_total
print(f"Train Accuracy: {train_acc * 100:.2f}%")

Train Accuracy: 92.34%


In [28]:
model.eval()
correct=0
total=0
with torch.no_grad():
  for (images,labels) in val_loader:
    output=model(images)
    _,pred=torch.max(output,1)
    correct+=(pred==labels).sum().item()
    total+=labels.size(0)
accuracy=100*correct/total
print(f"Total Accuracy : {accuracy:.2f}%")



Total Accuracy : 92.08%


In [30]:
model.eval()
correct=0
total=0
with torch.no_grad():
    for (images,labels) in test_loader:
        outputs=model(images)
        _,pred=torch.max(outputs,1)
        correct+=(pred==labels).sum().item()
        total+=labels.size(0)
accuracy=100*correct/total
print(f"Total Accuracy : {accuracy:.2f}%")


Total Accuracy : 92.14%


In [31]:
torch.save(model.state_dict(),'/kaggle/working/trash_classifier_resnet18.pth')
print("Model saved successfully!")

Model saved successfully!
